In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/DSS5104/results'
os.makedirs(SAVE_DIR, exist_ok=True)

!pip install -q rtdl==0.0.13 --no-deps
!pip install -q optuna==3.5.0

import rtdl, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"✅ rtdl: {rtdl.__version__}")
print(f"✅ optuna: {optuna.__version__}")
print(f"✅ Save dir: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ rtdl: 0.0.13
✅ optuna: 3.5.0
✅ Save dir: /content/drive/MyDrive/DSS5104/results


In [2]:
import numpy as np
import pandas as pd
import pickle, os, time, warnings, json
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import rtdl
from sklearn.datasets import fetch_california_housing, fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, roc_auc_score, f1_score)

SEEDS = [0, 42, 123]
N_TRIALS = 20
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device: {DEVICE}")

✅ Device: cuda


In [3]:
# California Housing
raw = fetch_california_housing(as_frame=True)
df_cal = raw.frame.copy()

# Adult Income
col_names = ['age','workclass','fnlwgt','education','education_num',
             'marital_status','occupation','relationship','race','sex',
             'capital_gain','capital_loss','hours_per_week','native_country','income']
df_a1 = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                    header=None, names=col_names, skipinitialspace=True, na_values='?')
df_a2 = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
                    header=None, names=col_names, skipinitialspace=True, na_values='?', skiprows=1)
df_adult = pd.concat([df_a1, df_a2], ignore_index=True)
df_adult['income'] = df_adult['income'].str.replace('.','',regex=False).str.strip()
df_adult['income'] = (df_adult['income'] == '>50K').astype(int)

# Covertype
raw_cov = fetch_covtype(as_frame=True)
df_cov = raw_cov.frame.copy()
df_cov = df_cov.sample(n=20_000, random_state=42).reset_index(drop=True)
df_cov['Cover_Type'] = df_cov['Cover_Type'] - 1

print("✅ Datasets loaded")
print(f"  California: {df_cal.shape}, Adult: {df_adult.shape}, Covertype: {df_cov.shape}")

✅ Datasets loaded
  California: (20640, 9), Adult: (48842, 15), Covertype: (20000, 55)


In [4]:
def preprocess_dataset(df, target_col, task, seed=42):
    df = df.copy()
    X = df.drop(columns=[target_col])
    y = df[target_col].values

    cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    for col in cat_cols:
        X[col] = X[col].fillna('__missing__')
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    for col in num_cols:
        X[col] = X[col].fillna(X[col].median())

    feature_cols = num_cols + cat_cols
    X = X[feature_cols].values.astype(np.float32)

    X_tr, X_temp, y_tr, y_temp = train_test_split(
        X, y, test_size=0.40, random_state=seed,
        stratify=y if task=='classification' else None)
    X_val, X_te, y_val, y_te = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed,
        stratify=y_temp if task=='classification' else None)

    n_num = len(num_cols)
    scaler = StandardScaler()
    X_tr[:, :n_num]  = scaler.fit_transform(X_tr[:, :n_num])
    X_val[:, :n_num] = scaler.transform(X_val[:, :n_num])
    X_te[:, :n_num]  = scaler.transform(X_te[:, :n_num])

    print(f"  Train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_te.shape}")
    print(f"  Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

    return (X_tr, X_val, X_te,
            y_tr.astype(np.float32) if task=='regression' else y_tr.astype(np.int64),
            y_val.astype(np.float32) if task=='regression' else y_val.astype(np.int64),
            y_te.astype(np.float32)  if task=='regression' else y_te.astype(np.int64),
            feature_cols, len(num_cols))

def evaluate(y_true, y_pred, y_prob, task):
    metrics = {}
    if task == 'regression':
        metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics['MAE']  = mean_absolute_error(y_true, y_pred)
        metrics['R2']   = r2_score(y_true, y_pred)
    elif task == 'classification':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob[:, 1])
        metrics['F1']       = f1_score(y_true, y_pred, average='binary')
    elif task == 'multiclass':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob,
                                             multi_class='ovr', average='macro')
        metrics['F1']       = f1_score(y_true, y_pred, average='macro')
    return metrics

print("✅ Functions defined")

✅ Functions defined


In [5]:
print("📦 Preprocessing...")

cal_splits   = preprocess_dataset(df_cal,   'MedHouseVal', 'regression')
adult_splits = preprocess_dataset(df_adult, 'income',      'classification')
cov_splits   = preprocess_dataset(df_cov,   'Cover_Type',  'multiclass')

datasets = {
    'california': {'splits': cal_splits,   'task': 'regression',    'n_classes': 1},
    'adult':      {'splits': adult_splits, 'task': 'classification', 'n_classes': 2},
    'covertype':  {'splits': cov_splits,   'task': 'multiclass',     'n_classes': 7},
}
print("✅ Done")

📦 Preprocessing...
  Train: (12384, 8), Val: (4128, 8), Test: (4128, 8)
  Numeric: 8, Categorical: 0
  Train: (29305, 14), Val: (9768, 14), Test: (9769, 14)
  Numeric: 6, Categorical: 8
  Train: (12000, 54), Val: (4000, 54), Test: (4000, 54)
  Numeric: 54, Categorical: 0
✅ Done


In [6]:
def train_fttransformer(X_tr, y_tr, X_val, y_val, task, n_classes,
                        n_num, d_token, n_heads, n_layers, ffn_factor,
                        dropout, lr, batch_size, max_epochs, patience, seed):

    torch.manual_seed(seed)
    n_features = X_tr.shape[1]
    n_cat = n_features - n_num

    # 构建模型：用 make_baseline 替代 make_default
    cat_cardinalities = (
        [int(X_tr[:, n_num + i].max()) + 2 for i in range(n_cat)]
        if n_cat > 0 else None
    )

    model = rtdl.FTTransformer.make_baseline(
        n_num_features=n_num,
        cat_cardinalities=cat_cardinalities,
        d_token=d_token,
        n_blocks=n_layers,
        attention_dropout=dropout,
        ffn_d_hidden=int(d_token * ffn_factor),
        ffn_dropout=dropout,
        residual_dropout=0.0,
        d_out=1 if task == 'regression' else n_classes,
    )
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss() if task == 'regression' else nn.CrossEntropyLoss()

    def get_tensors(X, y):
        X_num = torch.tensor(X[:, :n_num], dtype=torch.float32).to(DEVICE)
        X_cat = torch.tensor(X[:, n_num:], dtype=torch.long).to(DEVICE) if n_cat > 0 else None
        y_t   = torch.tensor(y, dtype=torch.float32 if task=='regression' else torch.long).to(DEVICE)
        return X_num, X_cat, y_t

    X_tr_num,  X_tr_cat,  y_tr_t  = get_tensors(X_tr,  y_tr)
    X_val_num, X_val_cat, y_val_t = get_tensors(X_val, y_val)

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(max_epochs):
        model.train()
        idx = torch.randperm(len(X_tr_num))
        for start in range(0, len(idx), batch_size):
            b = idx[start:start+batch_size]
            xc = X_tr_cat[b] if X_tr_cat is not None else None
            out = model(X_tr_num[b], xc).squeeze(-1)
            loss = criterion(out, y_tr_t[b])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            out_val = model(X_val_num, X_val_cat).squeeze(-1)
            val_loss = criterion(out_val, y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    return model

def predict_fttransformer(model, X, n_num, task, n_classes, batch_size=1024):
    model.eval()
    n_cat = X.shape[1] - n_num
    all_out = []

    for start in range(0, len(X), batch_size):
        xb = X[start:start+batch_size]
        xn = torch.tensor(xb[:, :n_num], dtype=torch.float32).to(DEVICE)
        xc = torch.tensor(xb[:, n_num:], dtype=torch.long).to(DEVICE) if n_cat > 0 else None
        with torch.no_grad():
            out = model(xn, xc).squeeze(-1)
        all_out.append(out.cpu().numpy())

    out = np.concatenate(all_out, axis=0)

    if task == 'regression':
        return out, None
    else:
        probs = torch.softmax(torch.tensor(out), dim=-1).numpy()
        preds = np.argmax(probs, axis=1)
        return preds, probs

print("✅ FT-Transformer functions defined")

✅ FT-Transformer functions defined


In [7]:
def run_fttransformer(dataset_name, datasets, seeds, n_trials=20):
    d = datasets[dataset_name]
    task = d['task']
    X_tr, X_val, X_te, y_tr, y_val, y_te, feat_names, n_num = d['splits']
    n_classes = d['n_classes']

    all_metrics = []
    best_params_list = []

    for seed in seeds:
        print(f"\n  🌱 seed={seed}")
        torch.manual_seed(seed)
        np.random.seed(seed)

        def objective(trial):
            d_token  = trial.suggest_categorical('d_token', [64, 128, 192])
            n_heads  = trial.suggest_categorical('n_heads', [4, 8])
            n_layers = trial.suggest_int('n_layers', 2, 4)
            ffn_factor = trial.suggest_float('ffn_factor', 1.5, 4.0)
            dropout  = trial.suggest_float('dropout', 0.0, 0.3)
            lr       = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
            batch_size = trial.suggest_categorical('batch_size', [256, 512])

            # d_token 必须能被 n_heads 整除
            if d_token % n_heads != 0:
                raise optuna.exceptions.TrialPruned()

            model = train_fttransformer(
                X_tr, y_tr, X_val, y_val,
                task=task, n_classes=n_classes, n_num=n_num,
                d_token=d_token, n_heads=n_heads, n_layers=n_layers,
                ffn_factor=ffn_factor, dropout=dropout, lr=lr,
                batch_size=batch_size, max_epochs=50, patience=10,
                seed=seed)

            y_pred, y_prob = predict_fttransformer(
                model, X_val, n_num, task, n_classes)

            if task == 'regression':
                return np.sqrt(mean_squared_error(y_val, y_pred))
            else:
                return -accuracy_score(y_val, y_pred)

        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

        best = study.best_params
        best_params_list.append(best)
        print(f"    Best params: {best}")

        # 最优参数在 train+val 上重训
        X_fit = np.concatenate([X_tr, X_val])
        y_fit = np.concatenate([y_tr, y_val])

        final_model = train_fttransformer(
            X_fit, y_fit, X_val, y_val,
            task=task, n_classes=n_classes, n_num=n_num,
            d_token=best['d_token'], n_heads=best['n_heads'],
            n_layers=best['n_layers'], ffn_factor=best['ffn_factor'],
            dropout=best['dropout'], lr=best['lr'],
            batch_size=best['batch_size'],
            max_epochs=100, patience=15, seed=seed)

        y_pred, y_prob = predict_fttransformer(
            final_model, X_te, n_num, task, n_classes)

        m = evaluate(y_te, y_pred, y_prob, task)
        m['seed'] = seed
        all_metrics.append(m)
        print(f"    Test metrics: {m}")

        # 立即保存
        save_path = f'{SAVE_DIR}/ftt_{dataset_name}_seed{seed}.pkl'
        with open(save_path, 'wb') as f:
            pickle.dump({'metrics': m, 'best_params': best}, f)
        print(f"    💾 Saved to {save_path}")

    df_m = pd.DataFrame(all_metrics).drop(columns='seed')
    summary = {col: f"{df_m[col].mean():.4f} ± {df_m[col].std():.4f}"
               for col in df_m.columns}
    return summary, best_params_list

print("✅ run_fttransformer() defined")

✅ run_fttransformer() defined


In [11]:
print("="*55)
print("🔷 FT-Transformer — CALIFORNIA HOUSING")
print("="*55)
t0 = time.time()

cal_summary, cal_params = run_fttransformer('california', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {cal_summary}")

with open(f'{SAVE_DIR}/ftt_california_summary.json', 'w') as f:
    json.dump(cal_summary, f)

🔷 FT-Transformer — CALIFORNIA HOUSING

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 128, 'n_heads': 4, 'n_layers': 3, 'ffn_factor': 1.9969197824660223, 'dropout': 0.08306317604150071, 'lr': 0.00023568842432698144, 'batch_size': 256}
    Test metrics: {'RMSE': np.float64(0.4801388670297523), 'MAE': 0.2980424463748932, 'R2': 0.831865668296814, 'seed': 0}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_california_seed0.pkl

  🌱 seed=42


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 128, 'n_heads': 4, 'n_layers': 3, 'ffn_factor': 3.8855742605278287, 'dropout': 0.10360384951623179, 'lr': 0.00010109743096172435, 'batch_size': 256}
    Test metrics: {'RMSE': np.float64(0.48109618366910684), 'MAE': 0.300184041261673, 'R2': 0.8311945199966431, 'seed': 42}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_california_seed42.pkl

  🌱 seed=123


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 192, 'n_heads': 4, 'n_layers': 4, 'ffn_factor': 3.0317983992935686, 'dropout': 0.09479427799727087, 'lr': 0.00024063043918360821, 'batch_size': 256}
    Test metrics: {'RMSE': np.float64(0.5083667371541264), 'MAE': 0.31321078538894653, 'R2': 0.8115149140357971, 'seed': 123}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_california_seed123.pkl

✅ Done in 31.9 min
Summary: {'RMSE': '0.4899 ± 0.0160', 'MAE': '0.3038 ± 0.0082', 'R2': '0.8249 ± 0.0116'}


In [12]:
print("="*55)
print("🔷 FT-Transformer — ADULT INCOME")
print("="*55)
t0 = time.time()

adult_summary, adult_params = run_fttransformer('adult', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {adult_summary}")

with open(f'{SAVE_DIR}/ftt_adult_summary.json', 'w') as f:
    json.dump(adult_summary, f)

🔷 FT-Transformer — ADULT INCOME

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 64, 'n_heads': 4, 'n_layers': 3, 'ffn_factor': 1.5659949901799828, 'dropout': 0.06739729237225396, 'lr': 0.0005608777977143769, 'batch_size': 512}
    Test metrics: {'Accuracy': 0.8446105026102979, 'AUC': np.float64(0.8932999966961515), 'F1': 0.6613119143239625, 'seed': 0}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_adult_seed0.pkl

  🌱 seed=42


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 64, 'n_heads': 8, 'n_layers': 3, 'ffn_factor': 3.4274179498864026, 'dropout': 0.14813867890931723, 'lr': 0.0003332213575546234, 'batch_size': 256}
    Test metrics: {'Accuracy': 0.8531067663015662, 'AUC': np.float64(0.9014435803403287), 'F1': 0.674233825198638, 'seed': 42}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_adult_seed42.pkl

  🌱 seed=123


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 128, 'n_heads': 4, 'n_layers': 3, 'ffn_factor': 2.720691932332642, 'dropout': 0.04464071428483773, 'lr': 0.00024208599853206932, 'batch_size': 512}
    Test metrics: {'Accuracy': 0.8335551233493704, 'AUC': np.float64(0.8833290797722855), 'F1': 0.6364042933810375, 'seed': 123}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_adult_seed123.pkl

✅ Done in 43.6 min
Summary: {'Accuracy': '0.8438 ± 0.0098', 'AUC': '0.8927 ± 0.0091', 'F1': '0.6573 ± 0.0192'}


In [8]:
print("="*55)
print("🔷 FT-Transformer — COVERTYPE")
print("="*55)
t0 = time.time()

cov_summary, cov_params = run_fttransformer('covertype', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {cov_summary}")

with open(f'{SAVE_DIR}/ftt_covertype_summary.json', 'w') as f:
    json.dump(cov_summary, f)

🔷 FT-Transformer — COVERTYPE

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 64, 'n_heads': 8, 'n_layers': 4, 'ffn_factor': 2.3636905461638147, 'dropout': 0.03912512709715518, 'lr': 0.0007182571121783259, 'batch_size': 512}
    Test metrics: {'Accuracy': 0.8495, 'AUC': np.float64(0.9743845290069364), 'F1': 0.7712411248385578, 'seed': 0}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_covertype_seed0.pkl

  🌱 seed=42


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 192, 'n_heads': 4, 'n_layers': 4, 'ffn_factor': 3.9855932144964306, 'dropout': 0.05877869656555358, 'lr': 0.00014797693982820188, 'batch_size': 512}
    Test metrics: {'Accuracy': 0.86625, 'AUC': np.float64(0.9695985908854662), 'F1': 0.7874198776693202, 'seed': 42}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_covertype_seed42.pkl

  🌱 seed=123


  0%|          | 0/20 [00:00<?, ?it/s]

    Best params: {'d_token': 128, 'n_heads': 4, 'n_layers': 2, 'ffn_factor': 3.360765951151003, 'dropout': 0.025042815564666254, 'lr': 0.00024368174285216863, 'batch_size': 512}
    Test metrics: {'Accuracy': 0.8615, 'AUC': np.float64(0.9730599350208747), 'F1': 0.788051407052681, 'seed': 123}
    💾 Saved to /content/drive/MyDrive/DSS5104/results/ftt_covertype_seed123.pkl

✅ Done in 153.0 min
Summary: {'Accuracy': '0.8591 ± 0.0086', 'AUC': '0.9723 ± 0.0025', 'F1': '0.7822 ± 0.0095'}


In [10]:
import pickle, json
import pandas as pd

# 从 Google Drive 读取各数据集结果
def load_summary(dataset_name):
    all_metrics = []
    for seed in [0, 42, 123]:
        path = f'{SAVE_DIR}/ftt_{dataset_name}_seed{seed}.pkl'
        with open(path, 'rb') as f:
            r = pickle.load(f)
        m = r['metrics']
        m['seed'] = seed
        all_metrics.append(m)
    df_m = pd.DataFrame(all_metrics).drop(columns='seed')
    summary = {col: f"{df_m[col].mean():.4f} ± {df_m[col].std():.4f}"
               for col in df_m.columns}
    return summary

cal_summary = load_summary('california')
adult_summary = load_summary('adult')
cov_summary = load_summary('covertype')

ftt_results = {
    'california': cal_summary,
    'adult':      adult_summary,
    'covertype':  cov_summary,
}

with open(f'{SAVE_DIR}/ftt_results.pkl', 'wb') as f:
    pickle.dump(ftt_results, f)

rows = []
for ds, summary in ftt_results.items():
    for metric, val in summary.items():
        rows.append({'dataset': ds, 'model': 'FT-Transformer',
                     'metric': metric, 'value': val})

df_ftt = pd.DataFrame(rows)
df_ftt.to_csv(f'{SAVE_DIR}/ftt_results.csv', index=False)

print("="*55)
print("📊 FT-TRANSFORMER RESULTS SUMMARY")
print("="*55)
print(df_ftt.to_string())
print("\n✅ All results saved!")

📊 FT-TRANSFORMER RESULTS SUMMARY
      dataset           model    metric            value
0  california  FT-Transformer      RMSE  0.4899 ± 0.0160
1  california  FT-Transformer       MAE  0.3038 ± 0.0082
2  california  FT-Transformer        R2  0.8249 ± 0.0116
3       adult  FT-Transformer  Accuracy  0.8438 ± 0.0098
4       adult  FT-Transformer       AUC  0.8927 ± 0.0091
5       adult  FT-Transformer        F1  0.6573 ± 0.0192
6   covertype  FT-Transformer  Accuracy  0.8591 ± 0.0086
7   covertype  FT-Transformer       AUC  0.9723 ± 0.0025
8   covertype  FT-Transformer        F1  0.7822 ± 0.0095

✅ All results saved!
